### ESRGAN - Enhanced Super-Resolution Generative Adversarial Network

<small>

U ovom delu implementiran je **ESRGAN** (*Enhanced Super-Resolution Generative Adversarial Network*) kao **treći i najnapredniji model** za **super-rezoluciju slika**.

- ESRGAN predstavlja dalje **unapređenje SRResNet arhitekture**, pri čemu se standardni rezidualni blokovi zamenjuju **RRDB blokovima** (*Residual-in-Residual Dense Blocks*).
- Za razliku od prethodnog **SRResNet modela**, cilj ESRGAN-a nije samo smanjenje **pixel-wise greške**, već i generisanje **vizuelno realističnijih i detaljnijih rekonstrukcija**.
- Tokom treniranja koriste se **pixel loss**, **perceptual loss** i **adversarial loss**, čime se model podstiče da pored globalne strukture slike bolje rekonstruiše **teksture, ivice i sitne detalje**.
- Kao i kod SRResNet-a, **LR slike** dimenzija $32\times32$ direktno se prosleđuju modelu, dok generator vrši povećavanje rezolucije za faktor $\times4$ i generiše slike dimenzija $128\times128$.
- **Performanse modela** procenjene su pomoću **PSNR** i **SSIM metrika**, uz poređenje rezultata **ESRGAN**-a sa **BICUBIC interpolacijom**.

</small>

In [ ]:
import random
import numpy as np
import torch

SEED = 48
random.seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

<small>

> Radi obezbeđivanja **reproduktivnosti rezultata**, postavljen je **fiksni random seed** ($SEED = 48$) za biblioteke $random$ i $NumPy$. <br>
> Na ovaj način se kontrolišu slučajni procesi, poput inicijalizacije težina modela i mešanja podataka tokom treniranja, tako da se pri ponovnom pokretanju koda pod istim uslovima dobijaju isti ili što konzistentniji rezultati.

</small>

#### I : Učitavanje biblioteka i pripremljenih trening i validacionih skupova

<small>

Učitavaju se prethodno pripremljeni **LR** i **HR podaci** iz $02\_Data\_Preparation.ipynb$ fajla. **LR slike** imaju dimenzije $32×32$, dok **HR slike** imaju dimenzije $128×128$ i predstavljaju **ciljne vrednosti** za treniranje modela.

</small>

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

X_train = np.load("../data/X_train.npy")
y_train = np.load("../data/y_train.npy")

X_val = np.load("../data/X_val.npy")
y_val = np.load("../data/y_val.npy")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (8000, 32, 32, 3)
y_train: (8000, 128, 128, 3)
X_val: (1000, 32, 32, 3)
y_val: (1000, 128, 128, 3)


<small>

> **Trening skup** sadrži $8.000$ parova, dok **validacioni skup** sadrži $1.000$ parova.<br>
> **LR** slike imaju dimenzije $32×32×3$, a odgovarajuće **HR** slike $128×128×3$, pri čemu $3$ **kanala** predstavljaju **RGB komponente** slike.

</small>

#### II : Normalizacija vrednosti piksela trening i validacionih podataka

<small>

Radi stabilnijeg i efikasnijeg treniranja neuronske mreže, vrednosti piksela **normalizuju** se sa **originalnog intervala** $[0, 255]$ na **interval** $[0, 1]$.

- Ovim postupkom se **vrednosti piksela** svode na **manji** i **ujednačen numerički opseg**, čime se **olakšava proces učenja** i doprinosi **stabilnijem radu modela**.
- Normalizacija nije izvršena tokom pripreme podataka, jer **SRCNN** zahteva prethodno **povećavanje LR slika BICUBIC interpolacijom** pre **normalizacije**, dok **SRResNet** i **ESRGAN** koriste LR slike u njihovoj **originalnoj rezoluciji**.

</small>

In [ ]:
X_train = X_train.astype("float32") / 255.0
y_train = y_train.astype("float32") / 255.0

X_val = X_val.astype("float32") / 255.0
y_val = y_val.astype("float32") / 255.0

print("X_train min:", X_train.min())
print("X_train max:", X_train.max())

print("y_train min:", y_train.min())
print("y_train max:", y_train.max())

X_train min: 0.0
X_train max: 1.0
y_train min: 0.0
y_train max: 1.0


<small>

> Nakon normalizacije, **vrednosti piksela** u **trening skupu** za LR i HR slike nalaze se u očekivanom intervalu $[0, 1]$, što potvrđuje da je **normalizacija** uspešno izvršena.

</small>

#### III : Konverzija podataka u tenzore i priprema za treniranje

<small>

Nakon prethodne obrade, podaci se pripremaju za unos u **ESRGAN model** implementiran u PyTorch-u.

- **Prilagođavanje formata**: LR i HR slike se **transponuju** iz formata $(H, W, C)$ u $(C, H, W)$, jer PyTorch **konvolutivni slojevi** očekuju da se **broj kanala** nalazi **pre dimenzija slike**.
- **Konverzija u tenzore**: $NumPy$ nizovi se pretvaraju u $torch.Tensor$ objekte kako bi mogli direktno da se koriste u PyTorch modelu.
- **Formiranje skupova**: Odvojeno se formiraju $train\_dataset$ i $val\_dataset$, koji sadrže **ulazne BICUBIC slike** i odgovarajuće **originalne HR slike**.
- **Batch obrada**: Podaci se organizuju u **batch**-eve od $16$ slika, čime se omogućava **istovremena obrada** više uzoraka tokom treniranja.
- **Mešanje podataka**: Trening podaci se nasumično mešaju ($shuffle=True$) kako bi se **smanjila zavisnost modela** od **redosleda uzoraka**, dok se kod validacionog skupa redosled zadržava ($shuffle=False$) radi konzistentne evaluacije.

</small>

In [4]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train = np.transpose(X_train, (0, 3, 1, 2))
y_train = np.transpose(y_train, (0, 3, 1, 2))

X_val = np.transpose(X_val, (0, 3, 1, 2))
y_val = np.transpose(y_val, (0, 3, 1, 2))

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

X_train: (8000, 3, 32, 32)
y_train: (8000, 3, 128, 128)


<small>

> Nakon transponovanja, **trening podaci** imaju oblik $(8000, 3, 32, 32)$, tj. $(8000, 3, 128, 128)$, što odgovara $PyTorch$ formatu (**broj slika**, **broj kanala**, **visina**, **širina**). <br>
> Za razliku od SRCNN modela, nije potrebno prethodno povećavanje LR slika BICUBIC interpolacijom. Kao i kod **SRResNet-a**, povećavanje prostorne rezolucije vrši se **unutar same mreže**.

<small>

#### IV : Arhitektura ESRGAN sistema

<small>

**ESRGAN** (*Enhanced Super-Resolution Generative Adversarial Network*) predstavlja **GAN arhitekturu namenjenu super-rezoluciji slika**, čiji je cilj rekonstrukcija slike **visoke rezolucije (HR)** na osnovu odgovarajuće slike **niske rezolucije (LR)**. Za razliku od klasičnih konvolucionih modela za super-rezoluciju, **ESRGAN** ne pokušava samo da minimizuje razliku između vrednosti piksela rekonstruisane i originalne slike, već je usmeren i na generisanje **vizuelno uverljivih tekstura i sitnih detalja**.

**ESRGAN sistem** se zasniva na **dve neuronske mreže — generatoru** (*Generator*) i **diskriminatoru** (*Discriminator*), koje se tokom treninga optimizuju kroz međusobno suprotstavljene ciljeve.

- **Generator (G)** prima **LR sliku** i na osnovu nje generiše odgovarajuću **SR** (*Super-Resolved*) **sliku** veće rezolucije. Osnovu generatora čine **RRDB** (*Residual-in-Residual Dense Block*) blokovi, koji omogućavaju efikasno **izdvajanje** i **kombinovanje karakteristika slike** na različitim nivoima. Nakon izdvajanja karakteristika, vrši se **uvećanje prostorne rezolucije** (*upsampling*) kako bi se dobila izlazna slika željenih dimenzija.
- **Diskriminator (D)** tokom treninga dobija dve vrste slika: **stvarne HR slike** iz skupa podataka i **SR slike generisane generatorom**. Njegov zadatak je da ***nauči** da **razlikuje stvarne od generisanih slika**. Na taj način diskriminator pruža generatoru povratnu informaciju koja ga usmerava ka stvaranju sve **realističnijih rezultata**.

**Generator** i **diskriminator** zajedno formiraju **adversarijalni sistem**: **generator** nastoji da **proizvede SR slike** koje će biti što **sličnije** stvarnim **HR slikama**, dok **diskriminator** nastoji da ih **pravilno razlikuje**. Njihovim **zajedničkim treningom generator postepeno uči** ne samo **rekonstrukciju osnovne strukture slike**, već i **tekstura, ivica i finih detalja** koji su značajni za perceptivni kvalitet rezultata.

Pored same arhitekture generatora i diskriminatora, važan deo **ESRGAN** sistema predstavlja i način definisanja **funkcije gubitka generatora**, koja kombinuje informacije o **razlici između rekonstruisane i originalne slike**, njihovim **perceptivnim karakteristikama** i **uspešnosti generatora** u odnosu na diskriminator. Na taj način **ESRGAN** pravi **kompromis** između **numeričke tačnosti rekonstrukcije** i **vizuelnog realizma generisane slike**.

**Tok obrade** može se pojednostavljeno **predstaviti** kao:

 1) **LR slika → Generator (RRDB blokovi + upsampling) → SR slika**

dok se **tokom treninga generisana SR slika**, zajedno sa **odgovarajućom stvarnom HR slikom**, **prosleđuje diskriminatoru**:

 2) **SR / HR slika → Diskriminator → procena realnosti slike**

Nakon **završenog treninga**, za samu **rekonstrukciju** novih slika **koristi** se **generator**, dok je **diskriminator** prvenstveno potreban tokom **procesa treniranja**.

</small>

##### 1\) Arhitektura ESRGAN generatora

<small>

**Generator** predstavlja **centralni** deo **ESRGAN sistema** i njegov osnovni zadatak je da iz ulazne slike **niske rezolucije (LR)** generiše odgovarajuću sliku **visoke rezolucije (SR)**. Generator postepeno **izdvaja karakteristike** ulazne slike, **obrađuje ih** kroz **veliki broj povezanih konvolucionih blokova**, a zatim **povećava prostornu rezoluciju slike** do željenih dimenzija.

**Arhitektura generatora** može se podeliti na **tri glavna dela**:

1. **Početni konvolucioni sloj**  

   - Ulazna **LR slika** najpre prolazi kroz **konvolucioni sloj** čiji je zadatak da **iz originalnih RGB vrednosti izdvoji početne karakteristike slike**, kao što su **ivice, prelazi, oblici i teksture**. 
   - Dobijene **mape karakteristika** predstavljaju **ulaz** u **glavni deo generatora**.

2. **RRDB blokovi** (*Residual-in-Residual Dense Blocks*) 

   - **Glavni deo ESRGAN generatora** čini niz **RRDB blokova**, koji omogućavaju **učenje složenijih karakteristika slike**.
   - Svaki **RRDB blok** sadrži **više međusobno povezanih** **RDB**(*Residual Dense Block*) **blokova**.
   - Unutar ovih blokova koriste se **dense veze**, tako da se **izlazi prethodnih slojeva prosleđuju narednim slojevima**.
   - Na ovaj način **mreža** može **ponovo da koristi** već **izdvojene karakteristike** i **kombinuje informacije sa različitih nivoa**.
   - Pored **dense veza**, koriste se i **rezidualne** (*skip*) **veze**, kojima se **ulaz određenog bloka direktno dodaje njegovom izlazu**.
   - One omogućavaju **efikasniji protok informacija** i **gradijenata** kroz **duboku mrežu** i **olakšavaju njeno treniranje**.
   - Naziv **Residual-in-Residual** potiče upravo od **postojanja rezidualnih veza na više nivoa** — **rezidualne strukture** postoje **unutar pojedinačnih blokova**, ali i **oko grupe tih blokova**.

3. **Upsampling i izlazni slojevi** 

   - Nakon prolaska kroz **RRDB blokove**, dobijene **mape karakteristika** prolaze kroz **upsampling slojeve**, čiji je zadatak **povećanje prostornih dimenzija slike**. 
   - Kod **super-rezolucije** sa **faktorom ×4**, **rezolucija** se tipično **povećava** u **dva uzastopna koraka** od **×2**.
   - Nakon **povećanja rezolucije**, **završni konvolucioni slojevi** pretvaraju **izdvojene mape karakteristika** u konačnu **RGB sliku sa tri kanala**, koja predstavlja generisanu **SR sliku**.

Pojednostavljeno, **tok podataka** kroz **ESRGAN generator** može se predstaviti kao: **LR slika → početna konvolucija → RRDB blokovi → upsampling → završne konvolucije → SR slika** <br>
Ključna karakteristika **ESRGAN generatora** je upravo upotreba **RRDB blokova**, koji omogućavaju **izgradnju veoma duboke mreže** i **efikasno učenje finih detalja** i **tekstura** potrebnih za dobijanje **vizuelno kvalitetnih slika visoke rezolucije**.

</small>

1.1\) Konstrukcija **RDB** (*Residual Dense Block*) bloka:

<small>

**Residual Dense Block (RDB)** predstavlja **osnovnu gradivnu komponentu RRDB bloka**, koja čini **centralni deo ESRGAN generatora**. 

- Njegova **uloga** je da omogući **efikasno izdvajanje** i **očuvanje karakteristika slike** kombinovanjem **gustih (dense) veza** i **rezidualnog učenja**.
- Sastoji se od niza **konvolucionih slojeva**, pri čemu svaki **naredni sloj** kao **ulaz** dobija **mape karakteristika svih prethodnih slojeva**.
- Ove **mape** se **spajaju** po **dimenziji kanala** i na taj način **informacije izdvojene u ranijim slojevima** ostaju **dostupne i dubljim slojevima mreže**.
- Pored **dense veza**, **RDB** koristi i **rezidualnu vezu**, kojom se **izlaz bloka**, **skaliran faktorom** $0.2$, dodaje **njegovom originalnom ulazu**. 
- Ovakva organizacija doprinosi **stabilnijem učenju duboke mreže** i **olakšava prenos informacija** i **gradijenata** kroz **veliki broj slojeva**.

U nastavku je implementiran **RDB blok sa pet konvolucionih slojeva**, pri čemu se nakon prva četiri sloja primenjuje **LeakyReLU** aktivaciona funkcija.

</small>

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class RDB(nn.Module):

    def __init__(self, channels=64, growth_channels=32):

        super(RDB,self).__init__()

        self.conv1 = nn.Conv2d(in_channels=channels, out_channels=growth_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=channels + growth_channels, out_channels=growth_channels, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=channels + 2*growth_channels, out_channels = growth_channels, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=channels + 3*growth_channels, out_channels = growth_channels, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(in_channels=channels + 4*growth_channels, out_channels = channels, kernel_size=3, padding=1)

    def forward(self,x):

        x1 = self.conv1(x)
        x1 = F.leaky_relu(x1, negative_slope=0.2, inplace=True)

        x2 = torch.cat([x, x1], dim=1)
        x2 = self.conv2(x2)
        x2 = F.leaky_relu(x2, negative_slope=0.2, inplace=True)

        x3 = torch.cat([x, x1, x2], dim=1)
        x3 = self.conv3(x3)
        x3 = F.leaky_relu(x3, negative_slope=0.2, inplace=True)

        x4 = torch.cat([x, x1, x2, x3], dim=1)
        x4 = self.conv4(x4)
        x4 = F.leaky_relu(x4, negative_slope=0.2, inplace=True)

        x5 = torch.cat([x, x1, x2, x3, x4], dim=1)
        x5 = self.conv5(x5)

        return x + 0.2 * x5

<small>

> Vrednost $channels = 64$ predstavlja broj **feature mapa** koje ulaze u RDB blok, dok $growth\_channels = 32$ određuje broj **novih feature mapa** koje generiše svaki od prva četiri konvoluciona sloja. <br>
> Ove vrednosti odgovaraju **standardnoj konfiguraciji ESRGAN arhitekture** i predstavljaju kompromis između sposobnosti mreže da izdvoji dovoljno veliki broj karakteristika slike i njene računske složenosti. 

> Zbog **dense povezivanja**, svaki naredni konvolucioni sloj dobija **originalnih 64 kanala**, zajedno sa po **32 nova kanala** iz svakog prethodnog sloja. Na taj način broj ulaznih kanala raste redom: **64 → 96 → 128 → 160 → 192**. <br>
> Ovakvo povezivanje omogućava svakom sloju **direktan pristup prethodno izdvojenim karakteristikama**, čime se podstiče njihovo **ponovno korišćenje** i olakšava **protok informacija i gradijenata** kroz mrežu.<br>
> Sve konvolucije koriste **kernel veličine 3×3**, koji omogućava izdvajanje **lokalnih prostornih karakteristika** slike, poput ivica, tekstura i sitnih detalja. <br>
> Postavljanjem $padding = 1$ širina i visina feature mapa ostaju nepromenjene nakon konvolucije, što je neophodno za njihovo kasnije **spajanje ($torch.cat$) i rezidualno sabiranje**.

> Poslednji konvolucioni sloj generiše ponovo **64 feature mape**, čime se broj kanala vraća na početnu vrednost. 
> Na taj način njegov izlaz ima iste dimenzije kao originalni ulaz $x$, što omogućava formiranje **rezidualne veze**: $x + 0.2 * x5$.
> Faktor **0.2** smanjuje doprinos novoizračunatih rezidualnih karakteristika pre njihovog dodavanja originalnom ulazu i koristi se radi **stabilnijeg treniranja duboke ESRGAN mreže**.

</small>

1.2\) Konstrukcija **RRDB** (*Residual-in-Residual Dense Blocks*) blokova:

<small>

**RRDB** (*Residual-in-Residual Dense Block*) predstavlja **osnovni gradivni blok ESRGAN generatora** i zasniva se na **povezivanju više** prethodno definisanih **RDB blokova**.

- U ovoj implementaciji, jedan **RRDB blok formiraćemo povezivanjem tri RDB bloka**, što odgovara standardnoj **ESRGAN** arhitekturi.
- **Izlaz** svakog **RDB bloka** prosleđuje se kao **ulaz narednom**, čime se omogućava **postepeno izdvajanje** i **obrada** sve **složenijih karakteristika slike**.
- Svaki **RDB** već sadrži sopstvenu **lokalnu rezidualnu vezu**, dok se oko **povezanih RDB blokova** uvodi još jedna, **spoljašnja rezidualna veza**. 
- Na taj način dobija se *residual-in-residual* struktura, odnosno **rezidualna veza unutar rezidualne veze**.
- Nakon prolaska kroz sva tri RDB bloka, dobijeni **izlaz** se **skalira faktorom** $0.2$ i dodaje **originalnom ulazu RRDB bloka**. 
- Ovakvo **skaliranje** doprinosi **stabilnijem treniranju duboke mreže**.

Kombinovanjem **dense povezivanja unutar pojedinačnih RDB blokova** i **rezidualnih veza na dva nivoa**, omogućava se bolje **ponovno korišćenje izdvojenih karakteristika**, kao i efikasniji **protok informacija i gradijenata** kroz duboku mrežu.

</small>

In [ ]:
import torch.nn as nn

class RRDB(nn.Module):

    def __init__(self, channels=64, growth_channels=32):

        super(RRDB,self).__init__()

        self.rdb1 = RDB(channels=channels, growth_channels=growth_channels)
        self.rdb2 = RDB(channels=channels, growth_channels=growth_channels)
        self.rdb3 = RDB(channels=channels, growth_channels=growth_channels)
        
    def forward(self,x):

        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)

        return x + 0.2 * out

<small>

> U okviru **RRDB bloka** povezujemo **tri prethodno definisana RDB bloka** jedan za drugim, pri čemu se **izlaz** svakog **RDB bloka** prosleđuje kao **ulaz narednom**. <br>
> Svakom **RDB bloku** prosleđuju se iste vrednosti $channels = 64$ i $growth\_channels = 32$. <br>
> Kako svaki **RDB** prima i vraća **64 feature mape**, izlaz jednog bloka može direktno da se prosledi narednom, bez dodatnog prilagođavanja broja kanala.<br>
> Povezivanjem **više RDB blokova** omogućava se **postepeno izdvajanje složenijih karakteristika slike**, dok se dodavanjem **spoljašnje rezidualne veze** oko **RDB blokova**, koji već sadrže sopstvene **lokalne rezidualne veze**, formira **Residual-in-Residual** struktura. 

> Nakon prolaska kroz sva tri RDB bloka, dobijeni izlaz $out$ skalira se faktorom $0.2$ i dodaje originalnom ulazu $x$, odnosno **izlaz RRDB bloka** računa se kao $x + 0.2 \times out$. <br>
> Ovaj faktor predstavlja **residual scaling** i koristi se za **smanjenje doprinosa rezidualnog dela**, čime se doprinosi **stabilnijem treniranju duboke ESRGAN mreže**. 

</small>

1.3\) Konstrukcija ESRGAN generatora:

<small>

Nakon definisanja pojedinačnih **RDB** i **RRDB blokova**, moguće je konstruisati kompletan **ESRGAN generator** njihovim **povezivanjem sa početnim konvolucionim**, **upsampling*** i **završnim konvolucionim slojevima**. 

- U ovoj implementaciji koristi se **8 uzastopno povezanih RRDB blokova**, čime se dobija **redukovana varijanta originalne ESRGAN arhitekture**. 
- **Broj blokova** je **smanjen** radi **smanjenja računarske zahtevnosti i vremena treniranja**, uz zadržavanje **osnovne strukture** i **principa rada ESRGAN generatora**. 
- Ulazna **RGB slika sa 3 kanala** **početnom konvolucijom** se preslikava u **64 feature mape**, koje zatim **prolaze** kroz **niz RRDB blokova**. 
- Nakon njihove obrade, primenjuje se **dodatna konvolucija** i **rezidualna veza na nivou glavnog dela generatora**, kojom se **dobijene karakteristike** kombinuju sa **feature mapama** izdvojenim **početnom konvolucijom**. 
- Za realizaciju **×4 super-rezolucije**, prostorne **dimenzije feature mapa povećavaju se** u **dva uzastopna ×2 koraka**, nakon čega **završni konvolucioni slojevi** transformišu **64 feature mape** u **3 izlazna RGB kanala**, odnosno konačnu **SR sliku**. 

</small>

In [ ]:
import torch.nn as nn

class Generator(nn.Module):

    def __init__(self, in_channels=3, out_channels=3, channels=64, num_rrdb=8):

        super(Generator, self).__init__()

        self.conv_first = nn.Conv2d(in_channels=in_channels, out_channels=channels, kernel_size=3, padding=1)
        
        self.rrdb_blocks = nn.Sequential(*[RRDB(channels=channels) for _ in range(num_rrdb)])
        self.trunk_conv = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1)

        self.upconv1 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1)
        self.upconv2 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1)

        self.hr_conv = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1)
        self.conv_last = nn.Conv2d(in_channels=channels, out_channels=out_channels, kernel_size=3, padding=1)


    def forward(self, x):

        first_features = self.conv_first(x)

        trunk = self.rrdb_blocks(first_features)
        trunk = self.trunk_conv(trunk)

        x = first_features + trunk

        # 32x32 -> 64x64
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.upconv1(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        # 64x64 -> 128x128
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.upconv2(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        x = self.hr_conv(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        x = self.conv_last(x)

        return x

<small>

> **Početni konvolucioni sloj** pretvara ulaznu **RGB sliku sa $3$ kanala** u **$64$ feature mape**, čime se formira **početna reprezentacija slike** pogodna za dalju obradu kroz mrežu. <br>
> Dobijenih **$64$ feature mapa** zatim prolazi kroz **$8$ uzastopno povezanih RRDB blokova**, koji predstavljaju glavni deo generatora i omogućavaju dubinsku obradu i učenje složenijih karakteristika slike. <br>
> **Izlaz poslednjeg RRDB bloka** prolazi kroz dodatnu $3 \times 3$ **konvoluciju**, kojom se dodatno obrađuju karakteristike dobijene kroz glavni deo mreže, uz zadržavanje $64$ **kanala** i istih prostornih dimenzija. <br>
> **Izlaz glavnog dela generatora** zatim se **sabira** sa **feature mapama dobijenim početnom konvolucijom**, čime se formira **duga rezidualna veza** koja omogućava **direktan prenos početnih karakteristika kroz mrežu**. 

> Prvi *upsampling* korak povećava prostorne dimenzije feature mapa sa $32 \times 32$ na $64 \times 64$ pomoću *nearest-neighbor* interpolacije sa faktorom **$\times 2$**, nakon čega konvolucija i **LeakyReLU** dodatno obrađuju povećane feature mape. <br>
> Drugi *upsampling* korak na isti način povećava rezoluciju sa $64 \times 64$ na $128 \times 128$, čime se zajedno sa prethodnim korakom postiže željeni faktor super-rezolucije **$\times 4$**. 

> Nakon povećanja rezolucije, dodatna konvolucija sa **LeakyReLU** obrađuje **$64$ feature mape u visokoj rezoluciji**, pripremajući ih za konačnu rekonstrukciju slike. <br>
> Završni konvolucioni sloj transformiše **$64$ feature mape u $3$ izlazna kanala**, čime se formira konačna **RGB slika visoke rezolucije dimenzija** $128 \times 128$. 

</small>

##### 2\) Arhitektura ESRGAN diskriminatora

<small>

**Diskriminator** predstavlja drugi ključni deo **ESRGAN sistema** i njegov osnovni zadatak je da proceni **realističnost slike**, odnosno da razlikuje **originalne slike visoke rezolucije (HR)** od **generisanih slika visoke rezolucije (SR)** koje proizvodi generator. Za razliku od generatora, diskriminator **ne generiše novu sliku**, već postepeno **izdvaja i analizira karakteristike ulazne slike** kako bi procenio koliko ona odgovara stvarnim HR slikama.

**Arhitektura diskriminatora** može se podeliti na **tri glavna dela**:

1. **Početno izdvajanje karakteristika**

   - **Ulaz u diskriminator** predstavlja **HR ili generisana SR RGB slika**, koje imaju **iste prostorne dimenzije** kako bi diskriminator mogao da ih poredi pod **jednakim uslovima**.
   - **Ulazna slika** najpre prolazi kroz **konvolucione slojeve**, čiji je zadatak **izdvajanje osnovnih vizuelnih karakteristika**, kao što su **ivice, teksture, prelazi i lokalni detalji**.

2. **Dubinska obrada i smanjivanje prostornih dimenzija**

   - Glavni deo diskriminatora čini niz **konvolucionih slojeva** kroz koje se **broj feature mapa** postepeno **povećava**, dok se **visina i širina feature mapa postepeno smanjuju** korišćenjem konvolucija sa korakom $stride = 2$.
   - **Smanjivanjem prostornih dimenzija** mreža postepeno prelazi sa analize **lokalnih detalja** na **složenije** i **šire karakteristike slike**, dok **povećavanje broja kanala** omogućava predstavljanje **većeg broja različitih naučenih karakteristika**.
   - Nakon konvolucionih slojeva primenjuje se **LeakyReLU aktivaciona funkcija**, koja uvodi **nelinearnost** i omogućava diskriminatoru **učenje složenijih razlika** između **stvarnih** i **generisanih slika**.

3. **Završna procena realističnosti**

   - Nakon izdvajanja i dubinske obrade karakteristika, dobijene **feature mape** se transformišu u **izlaznu vrednost diskriminatora**, koja predstavlja **njegovu procenu realističnosti ulazne slike**.
   - U **ESRGAN**-u se koristi **relativistički pristup diskriminaciji**, pri čemu cilj nije samo **nezavisno određivanje** da li je **određena slika stvarna** ili **generisana**, već procena **da li stvarna HR slika izgleda realističnije od generisane SR slike i obrnuto**.

Pojednostavljeno, **tok podataka** kroz **ESRGAN diskriminator** može se predstaviti kao: **HR/SR slika → izdvajanje karakteristika → konvolucioni slojevi i smanjivanje prostornih dimenzija → završna obrada → procena realističnosti**. <br>

Tokom treninga, **diskriminator i generator uče suprotstavljene zadatke** — diskriminator nastoji da što bolje **razlikuje** stvarne **HR slike** od generisanih **SR slika**, dok **generator** nastoji da **proizvede SR slike** koje će **diskriminator proceniti kao što realističnije**, čime se podstiče rekonstrukcija **uverljivijih tekstura i finih detalja**.

</small>

2.1\) Konstrukcija ESRGAN diskriminatora:

<small>

**ESRGAN diskriminator** će biti implementiran kao **VGG-stil konvolutivna mreža**, sastavljena od niza konvolucionih blokova čiji se broj kanala postepeno povećava (**64 → 128 → 256 → 512**), dok se prostorne dimenzije feature mapa postepeno smanjuju korišćenjem konvolucija sa **korakom (stride) 2**.

- Svaki blok konvolucija-normalizacija-aktivacija sastoji se od **konvolucionog sloja**, **Batch Normalizacije** (osim prvog sloja, gde se BN izostavlja radi stabilnijeg početnog treninga) i **LeakyReLU** aktivacije.
- Naizmenično se koriste konvolucije sa $stride=1$ (koje ne menjaju prostorne dimenzije, već samo dodatno obrađuju karakteristike) i $stride=2$ (koje **prepolovljuju** visinu i širinu feature mapa).
- Ulazna **HR/SR slika** dimenzija $128\times128$ nakon četiri $stride=2$ konvolucije dobija prostorne dimenzije $8\times8$, uz $512$ kanala.
- Dobijene feature mape se **spljošte (flatten)** i prosleđuju kroz dva **potpuno povezana (fully connected) sloja**, koji **izlazne karakteristike** transformišu u **jednu skalarnu vrednost** — procenu realističnosti ulazne slike.
- Poslednji sloj **ne koristi sigmoid aktivaciju**, jer se kod **relativističkog diskriminatora** (RaGAN pristupa koji ESRGAN koristi) sirovi izlazi (*logits*) stvarne i generisane slike direktno upoređuju unutar funkcije gubitka.

</small>

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Discriminator(nn.Module):

    def __init__(self, in_channels=3, base_channels=64):

        super(Discriminator, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=base_channels, kernel_size=3, stride=1, padding=1)

        self.conv2 = nn.Conv2d(in_channels=base_channels, out_channels=base_channels, kernel_size=3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(base_channels)

        self.conv3 = nn.Conv2d(in_channels=base_channels, out_channels=base_channels * 2, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(base_channels * 2)

        self.conv4 = nn.Conv2d(in_channels=base_channels * 2, out_channels=base_channels * 2, kernel_size=3, stride=2, padding=1)
        self.bn4 = nn.BatchNorm2d(base_channels * 2)

        self.conv5 = nn.Conv2d(in_channels=base_channels * 2, out_channels=base_channels * 4, kernel_size=3, stride=1, padding=1)
        self.bn5 = nn.BatchNorm2d(base_channels * 4)

        self.conv6 = nn.Conv2d(in_channels=base_channels * 4, out_channels=base_channels * 4, kernel_size=3, stride=2, padding=1)
        self.bn6 = nn.BatchNorm2d(base_channels * 4)

        self.conv7 = nn.Conv2d(in_channels=base_channels * 4, out_channels=base_channels * 8, kernel_size=3, stride=1, padding=1)
        self.bn7 = nn.BatchNorm2d(base_channels * 8)

        self.conv8 = nn.Conv2d(in_channels=base_channels * 8, out_channels=base_channels * 8, kernel_size=3, stride=2, padding=1)
        self.bn8 = nn.BatchNorm2d(base_channels * 8)

        self.fc1 = nn.Linear(base_channels * 8 * 8 * 8, 1024)
        self.fc2 = nn.Linear(1024, 1)

    def forward(self, x):

        x = self.conv1(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        x = self.conv4(x)
        x = self.bn4(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        x = self.conv5(x)
        x = self.bn5(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        x = self.conv6(x)
        x = self.bn6(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        x = self.conv7(x)
        x = self.bn7(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        x = self.conv8(x)
        x = self.bn8(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        x = x.view(x.size(0), -1)

        x = self.fc1(x)
        x = F.leaky_relu(x, negative_slope=0.2, inplace=True)

        x = self.fc2(x)

        return x

<small>

> **Prvi konvolucioni sloj** preslikava ulaznu **RGB sliku sa 3 kanala** u **64 feature mape**, bez Batch Normalizacije, čime se izbegava normalizacija sirovih ulaznih vrednosti piksela u ranoj fazi mreže. <br>
> Naredni parovi konvolucija (**stride 1** pa **stride 2**) naizmenično **udvostručuju broj kanala** ($64\to128\to256\to512$) i **prepolovljuju prostorne dimenzije** feature mapa, pri čemu se posle svake konvolucije primenjuje **Batch Normalizacija** i **LeakyReLU** aktivacija sa $negative\_slope=0.2$.

> Nakon četiri koraka smanjivanja rezolucije ($stride=2$), ulazna slika dimenzija $128\times128$ svodi se na feature mape dimenzija $8\times8$ sa $512$ kanala, koje se zatim **spljošte** u vektor dužine $512 \times 8 \times 8 = 32768$. <br>
> Ovaj vektor prolazi kroz **fully connected sloj** sa $1024$ neurona i **LeakyReLU** aktivacijom, a zatim kroz **poslednji fully connected sloj** koji generiše **jedan izlazni logit** — procenu realističnosti ulazne slike, koja se dalje koristi u okviru **relativističke adversarijalne funkcije gubitka**.

</small>

##### 3\) Provera dimenzija generatora i diskriminatora

<small>

Pre definisanja funkcija gubitka i pokretanja treninga, potrebno je proveriti da li **generator** i **diskriminator** ispravno obrađuju ulazne podatke i da li generišu izlaze očekivanih dimenzija.

- **Generator** treba da od ulazne **LR slike** dimenzija $32\times32\times3$ generiše **SR sliku** dimenzija $128\times128\times3$, čime se dobija faktor uvećanja $\times4$.
- **Diskriminator** treba da od ulazne **HR/SR slike** dimenzija $128\times128\times3$ proizvede **jednu skalarnu vrednost po slici** u batch-u, koja predstavlja procenu njene realističnosti.

Provera se vrši prosleđivanjem **veštački generisanih (random) tenzora** kroz oba modela i ispisivanjem dimenzija dobijenih izlaza.

</small>

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator = Generator().to(device)
discriminator = Discriminator().to(device)

dummy_lr = torch.randn(2, 3, 32, 32).to(device)
dummy_hr = torch.randn(2, 3, 128, 128).to(device)

sr_output = generator(dummy_lr)
d_output = discriminator(dummy_hr)

print("LR ulaz:", dummy_lr.shape)
print("SR izlaz generatora:", sr_output.shape)
print("HR ulaz u diskriminator:", dummy_hr.shape)
print("Izlaz diskriminatora:", d_output.shape)

<small>

> **Generator** je od ulaznog tenzora dimenzija $(2, 3, 32, 32)$ generisao izlazni tenzor dimenzija $(2, 3, 128, 128)$, čime je potvrđeno da mreža ispravno vrši uvećanje rezolucije za faktor $\times4$, u skladu sa ciljanim dimenzijama HR slika. <br>
> **Diskriminator** je od ulaznog tenzora dimenzija $(2, 3, 128, 128)$ proizveo izlazni tenzor dimenzija $(2, 1)$, odnosno **po jedan logit za svaku sliku u batch-u**, što odgovara očekivanoj arhitekturi binarnog procenjivača realističnosti.

> Ovim je potvrđeno da su **generator** i **diskriminator** ispravno implementirani i spremni za dalju upotrebu u treningu.

</small>

#### V : Definisanje funkcija gubitka

<small>

Za razliku od **SRResNet** modela, koji se optimizuje isključivo na osnovu **pixel-wise greške**, **ESRGAN** koristi **kombinaciju tri funkcije gubitka**, čime se generator podstiče da rekonstruiše i **globalnu strukturu** i **perceptivno realistične detalje** slike:

1. **Pixel-wise (sadržajni) gubitak** — obezbeđuje osnovnu strukturnu i bojenu vernost SR slike u odnosu na HR sliku.
2. **Perceptualni gubitak** — poredi karakteristike SR i HR slika izdvojene unapred istreniranom **VGG mrežom**, umesto direktnog poređenja piksela, čime se podstiče **teksturalna i perceptivna sličnost**.
3. **Adversarijalni (relativistički) gubitak** — podstiče generator da proizvodi slike koje diskriminator procenjuje kao **realističnije od stvarnih HR slika**, čime se dodatno poboljšava vizuelni kvalitet rezultata.

**Ukupan gubitak generatora** predstavlja **težinsku kombinaciju** ova tri gubitka, dok se **diskriminator** trenira zasebno, na osnovu sopstvene adversarijalne funkcije gubitka.

</small>

##### 1) Pixel-wise (sadržajni) gubitak

<small>

**Pixel-wise gubitak** meri **prosečnu apsolutnu razliku** ($L1$) između vrednosti piksela **SR** i **HR** slike i obezbeđuje da generisana slika zadrži **osnovnu strukturu, boje i globalni izgled** originalne HR slike.

- Za razliku od **SRResNet-a**, koji koristi **MSE ($L2$) gubitak**, **ESRGAN** koristi **$L1$ gubitak**, jer manje penalizuje velika odstupanja i doprinosi generisanju **oštrijih**, manje zamućenih slika.

</small>

In [ ]:
pixel_criterion = nn.L1Loss()

##### 2) Perceptualni gubitak (VGG feature-based)

<small>

**Perceptualni gubitak** ne poredi direktno vrednosti piksela, već **karakteristike (feature mape)** SR i HR slika izdvojene pomoću **unapred istrenirane VGG19 mreže** na ImageNet skupu podataka.

- Poređenjem **dubljih feature mapa**, umesto sirovih piksela, gubitak postaje osetljiviji na **teksture, oblike i perceptivne karakteristike** slike, a manje na tačne vrednosti pojedinačnih piksela.
- Koriste se feature mape izdvojene **pre poslednje aktivacione funkcije** unutar VGG mreže, čime se, prema originalnom ESRGAN radu, dobijaju **oštrije i informativnije karakteristike**.
- Težine **VGG mreže** se **zamrzavaju** ($requires\_grad=False$), jer se mreža koristi isključivo kao **fiksni ekstraktor karakteristika**, a ne kao deo modela koji se trenira.

</small>

In [ ]:
import torchvision.models as models

class VGGFeatureExtractor(nn.Module):

    def __init__(self, layer_index=35):

        super(VGGFeatureExtractor, self).__init__()

        vgg19 = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features
        self.feature_extractor = nn.Sequential(*list(vgg19)[:layer_index]).eval()

        for param in self.feature_extractor.parameters():
            param.requires_grad = False

    def forward(self, x):
        return self.feature_extractor(x)


vgg_extractor = VGGFeatureExtractor().to(device)
perceptual_criterion = nn.L1Loss()

<small>

> Parametar $layer\_index=35$ određuje do kog sloja VGG19 mreže se izdvajaju karakteristike — odabran je sloj koji odgovara **konvolucionim aktivacijama pre poslednjeg max-pooling sloja** u okviru **conv5** bloka, u skladu sa preporukom iz originalnog ESRGAN rada. <br>
> Postavljanjem $eval()$ moda i $requires\_grad=False$ za sve parametre, obezbeđuje se da **VGG mreža ostane fiksna** tokom treninga i da se koristi isključivo za izdvajanje karakteristika, bez ažuriranja sopstvenih težina. <br>
> Perceptualni gubitak se zatim računa kao **$L1$ rastojanje** između **VGG karakteristika SR slike** i **VGG karakteristika HR slike**.

</small>

##### 3) Adversarijalni (relativistički) gubitak

<small>

**ESRGAN** koristi **relativistički prosečni GAN (RaGAN)** pristup, kod koga diskriminator ne procenjuje **apsolutnu realističnost** pojedinačne slike, već **relativnu realističnost** — odnosno koliko je **stvarna HR slika realističnija od prosečne generisane SR slike**, i obrnuto.

- **Gubitak diskriminatora** podstiče ga da **stvarnim HR slikama** dodeli **veću relativnu realističnost** u odnosu na **generisane SR slike**.
- **Gubitak generatora** podstiče ga da generiše **SR slike** kojima diskriminator dodeljuje **veću relativnu realističnost** u odnosu na stvarne HR slike, čime se generator **direktno takmiči** sa distribucijom stvarnih slika.
- Za implementaciju se koristi $BCEWithLogitsLoss$, s obzirom na to da diskriminator vraća **sirove logite**, bez sigmoid aktivacije.

</small>

In [ ]:
adversarial_criterion = nn.BCEWithLogitsLoss()

def relativistic_discriminator_loss(real_logits, fake_logits):

    real_loss = adversarial_criterion(real_logits - fake_logits.mean(0, keepdim=True), torch.ones_like(real_logits))
    fake_loss = adversarial_criterion(fake_logits - real_logits.mean(0, keepdim=True), torch.zeros_like(fake_logits))

    return (real_loss + fake_loss) / 2


def relativistic_generator_loss(real_logits, fake_logits):

    real_loss = adversarial_criterion(real_logits - fake_logits.mean(0, keepdim=True), torch.zeros_like(real_logits))
    fake_loss = adversarial_criterion(fake_logits - real_logits.mean(0, keepdim=True), torch.ones_like(fake_logits))

    return (real_loss + fake_loss) / 2

<small>

> Funkcija $relativistic\_discriminator\_loss$ podstiče diskriminator da **stvarnim slikama** ($real\_logits$) dodeli veću realističnost u odnosu na **prosek generisanih slika**, i obrnuto za generisane slike — čime uči da **razlikuje relativnu realističnost**, a ne apsolutnu klasifikaciju stvarno/lažno. <br>
> Funkcija $relativistic\_generator\_loss$ ima **suprotne ciljne vrednosti** (labels su zamenjeni), čime se generator podstiče da **prevari diskriminator** tako što će njegove SR slike delovati **realističnije od proseka stvarnih HR slika**.

</small>

#### VI : Inicijalizacija modela, optimizatora i hiperparametara

<small>

Pre pokretanja treninga, potrebno je definisati **hiperparametre** kojima se kontroliše proces optimizacije, kao i **optimizatore** za **generator** i **diskriminator**, koji se treniraju **odvojeno**, svaki sa sopstvenom funkcijom gubitka.

- Za oba modela koristi se **Adam optimizator**, koji je standardan izbor kod GAN arhitektura zbog svoje **stabilnosti** i **adaptivnog podešavanja koraka učenja** za svaki parametar.
- **Stopa učenja (learning rate)** postavljena je na relativno malu vrednost, uobičajenu za treniranje GAN modela, kako bi se izbegla **nestabilnost treninga** i pojava tzv. **mode collapse-a**.
- **Težinski koeficijenti** ($\lambda$) određuju **relativan doprinos** svake od tri komponente gubitka generatora ukupnom gubitku:
  - $\lambda_{pixel} = 1$ — dominantan doprinos, obezbeđuje **strukturnu vernost** SR slike.
  - $\lambda_{perceptual} = 1$ — doprinosi **teksturalnoj i perceptivnoj kvaliteti**.
  - $\lambda_{adv} = 0.005$ — mnogo manji doprinos, jer **adversarijalni gubitak** ima veću numeričku osetljivost i, ukoliko bi imao veću težinu, mogao bi da **destabilizuje trening** ili dovede do generisanja **artefakata**.
- Definiše se i **broj epoha** treninga, kao i **uređaj** (*GPU*, ukoliko je dostupan) na kome će se model trenirati.

</small>

In [ ]:
import torch.optim as optim

NUM_EPOCHS = 30
LEARNING_RATE = 1e-4

LAMBDA_PIXEL = 1.0
LAMBDA_PERCEPTUAL = 1.0
LAMBDA_ADV = 0.005

generator = Generator().to(device)
discriminator = Discriminator().to(device)
vgg_extractor = VGGFeatureExtractor().to(device)

optimizer_G = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.999))

<small>

> Parametri $betas=(0.9, 0.999)$ predstavljaju **standardne vrednosti** za Adam optimizator i kontrolišu brzinu **eksponencijalnog opadanja** prosečnih vrednosti **gradijenta** i **kvadrata gradijenta**, čime se doprinosi **stabilnijem i glatkijem** ažuriranju težina tokom treninga. <br>
> Odvajanjem **optimizatora generatora** ($optimizer\_G$) i **optimizatora diskriminatora** ($optimizer\_D$), omogućava se **nezavisno ažuriranje težina** svake od dve mreže, u skladu sa njihovim sopstvenim funkcijama gubitka.

</small>

#### VII : Treniranje ESRGAN modela